In [0]:
%python
import requests
from bs4 import BeautifulSoup
import os

url = "https://download.bls.gov/pub/time.series/pr/"
output_files_path="/Volumes/pr_population_volume/default/raw/data/pr_data/"
headers = {
    "User-Agent": "DataEng-Practice (BLS API client; contact: rakesh.khairnar3488@gmail.com)"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")
# Extract all file links
file_links = [
    link.get("href") for link in soup.find_all("a")
    if link.get("href") and not link.get("href").endswith("/")
]
print("Files found:", file_links)

for file_path in file_links:
    file_url = url + os.path.basename(file_path)
    print(f"Downloading {file_url} ...")

    file_resp = requests.get(file_url, headers=headers)

    # Save to Databricks volume
    target_path = f"{output_files_path}/{os.path.basename(file_path)}"
    dbutils.fs.put(target_path, file_resp.text, overwrite=True)

print("Download complete!")
